# v20 GPU PF 差し込み(7.191 ノート用・Kaggle提出可)

**目的**: 7.191 ノートの selector が使う **`run_pf_lik_ensemble_scales`(128seed CPU)** を、
あなたの **v20 GPU PF(r0, 物理ゲート)** に差し替える。GBM特徴用の `run_pf_ancc` は**原版のまま**(学習済みモデルを壊さない)。

## 使い方(Kaggle)
1. 7.191 ノートで **`SELECTOR_SCALES` と `run_pf_lik_ensemble_scales` が定義された後・test推論ループの前**に、
   下の **セル1(エンジン)→ セル2(差し替え)** をそのまま貼る。
2. GPU ランタイムを有効化(`torch.cuda.is_available()` が True)。
3. そのまま実行→提出。`run_pf_lik_ensemble_scales` は自動で GPU 版に置き換わる(同名・同戻り値)。

## 仕組み
- test 全坑井で v20 GPU PF を **1回走らせ**、per-seed を `SELECTOR_SCALES` で再加重して
  `{pf_scale_3/5/8/12, pf_mean}` を **坑井フィンガープリント(MD長/始終MD)でキャッシュ**。
- 差し替え関数は `hw` のフィンガープリントでキャッシュ参照(7.191 側のループ改変不要)。未ヒット時は原版CPUにフォールバック。

## 注意
- **selector のみ差し替え**(安全)。GBM特徴 `run_pf_ancc` は原版維持。
- PFパラメータは v20 バンク r0(物理ON, 32seed)。別PFにするなら `_R0_RAW` / `_GN_SEEDS` を編集。
- 効果は ノイズ床±0.2 と 硬い核で確認。

In [ ]:
# ===== セル1: v20 GPU PF エンジン(自己完結・7.191 のどのヘルパにも依存しない)=====
import torch, numpy as _np, pandas as _pd, glob as _glob, os as _os, time as _time
_GDEV = "cuda" if torch.cuda.is_available() else "cpu"; _GDT = torch.float32
print("v20 GPU PF inject | device:", _GDEV)

# ---- r0(v20 強PF)raw パラメータ(pf_bank round0)----
_R0_RAW = dict(n_particles=500, likelihood_scale=6.265873932344672, init_pos_std=1.0481897997156377,
    init_rate_std=0.004385968881669577, mom=0.9995659517817024, vn=0.0009771590366402883,
    pn=0.02193433099852571, rp=1.266960649917433, rr=0.0003377853906820054, resamp=0.05083194815648496,
    gr_sig_min=10.293696795145928, gr_sig_max=49.46806691781805, gr_sig_def=129.97956203556075,
    gr_sig_mult=3.412937310824599, tvt_clip_margin=159.57810929194702, rate_clip=0.0,
    gr_power=1.5371545258734838, hgr_smooth_r=1, tw_gr_smooth_r=0, jump_prob=0.002, jump_std=5.0,
    phys_sig=4.0313262188520795, use_phys=True)
_GN_SEEDS = 32          # ★あなたのPFの調整regime(submit_v20=32)。128(=7.191自身のPF用)は流用しない
_GCHUNK   = 8           # ★GPUバッチの坑井数。実バッチ=_GCHUNK×_GN_SEEDS。OOMなら 8→4→2 と下げる

def _mk_p(r, n_seeds):
    p = {}
    p["ANCC_N"]=int(r["n_particles"]); p["PF_N_SEEDS"]=int(n_seeds)
    p["PF_LIKELIHOOD_SCALE"]=float(r["likelihood_scale"]); p["PF_INIT_SPR"]=float(r["init_pos_std"])
    p["PF_INIT_V_STD"]=float(r["init_rate_std"]); p["PF_MOM"]=float(r["mom"]); p["PF_VN"]=float(r["vn"])
    p["PF_PN"]=float(r["pn"]); p["PF_ROUGH_P"]=float(r["rp"]); p["PF_ROUGH_V"]=float(r["rr"])
    p["PF_RESAMP"]=float(r["resamp"]); p["PF_GR_SIG_MIN"]=float(r["gr_sig_min"]); p["PF_GR_SIG_MAX"]=float(r["gr_sig_max"])
    p["PF_GR_SIG_DEF"]=float(r["gr_sig_def"]); p["PF_GR_SIG_MULT"]=float(r["gr_sig_mult"])
    p["PF_TVT_CLIP_MARGIN"]=float(r["tvt_clip_margin"]); p["PF_RATE_CLIP"]=float(r["rate_clip"])
    p["PF_GR_POWER"]=float(r["gr_power"]); p["PF_HGR_SMOOTH_R"]=int(r["hgr_smooth_r"]); p["PF_TW_GR_SMOOTH_R"]=int(r["tw_gr_smooth_r"])
    p["PF_JUMP_PROB"]=float(r["jump_prob"]); p["PF_JUMP_STD"]=float(r["jump_std"])
    p["ANCC_ALPHA"]=p["PF_MOM"]; p["ANCC_RN"]=p["PF_VN"]; p["ANCC_PN"]=p["PF_PN"]; p["ANCC_IR"]=p["PF_INIT_V_STD"]
    p["ANCC_IS"]=p["PF_INIT_SPR"]; p["ANCC_RP"]=p["PF_ROUGH_P"]; p["ANCC_RR"]=p["PF_ROUGH_V"]
    p["PF_USE_PHYS"]=bool(r["use_phys"]); p["PF_PHYS_SIG"]=float(r["phys_sig"])
    return p
_GP = _mk_p(_R0_RAW, _GN_SEEDS)

# ---- 自前ヘルパ(7.191 の _smooth/_grid/_gr_sig とシグネチャ衝突を避け独自定義)----
def _gsm(vals, fb, r):
    s = _pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    if int(r) <= 0: return s.to_numpy(_np.float32)
    return s.rolling(2*int(r)+1, center=True, min_periods=1).mean().to_numpy(_np.float32)
def _ggrid(tw_tvt, tw_gr, step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max()); g=_np.arange(tmin,tmax+step,step)
    return _np.interp(g,tw_tvt,tw_gr).astype(_np.float64), float(tmin), float(step)
def _ggrsig(gr_proc, kn_pos, ktvt, tw_tvt, tw_gr_pf, p):
    if len(kn_pos) < 20: return float(p["PF_GR_SIG_DEF"])
    resid = gr_proc[kn_pos] - _np.interp(ktvt, tw_tvt, tw_gr_pf)
    g = float(_np.nanstd(resid)); g = g if (_np.isfinite(g) and g>0) else float(p["PF_GR_SIG_DEF"])
    return float(_np.clip(g*p["PF_GR_SIG_MULT"], p["PF_GR_SIG_MIN"], p["PF_GR_SIG_MAX"]))

def _gbuild(hw, tw, p):
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    if len(kn) < 5 or len(ev) < 1: return None
    tw = tw.sort_values("TVT"); tw_tvt = tw["TVT"].to_numpy(_np.float64); tw_gr = tw["GR"].to_numpy(_np.float64)
    if len(tw_tvt) < 3: return None
    fb = float(_np.nanmean(tw_gr)) if _np.isfinite(_np.nanmean(tw_gr)) else 0.
    tw_gr_pf = _gsm(tw_gr, fb, p["PF_TW_GR_SMOOTH_R"]).astype(_np.float64)
    gr_full = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(fb).to_numpy(float)
    gr_proc = _gsm(gr_full, fb, p["PF_HGR_SMOOTH_R"]).astype(_np.float64)
    ev_pos = hw.index.get_indexer(ev.index); kn_pos = hw.index.get_indexer(kn.index)
    ls = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
    tail = kn.tail(30); dt=_np.diff(tail["TVT_input"].values); dz=_np.diff(tail["Z"].values); dm=_np.diff(tail["MD"].values); m=dm>0
    ir = float(_np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst = _ggrid(tw_tvt, tw_gr_pf)
    gs = _ggrsig(gr_proc, kn_pos, kn["TVT_input"].to_numpy(_np.float64), tw_tvt, tw_gr_pf, p)
    return dict(md=ev["MD"].to_numpy(_np.float64), z=ev["Z"].to_numpy(_np.float64),
                gr=gr_proc[ev_pos].astype(_np.float64), gg=gg, gmin=gmin, gst=gst, gs=gs, ls=ls, ir=ir, N=int(p["ANCC_N"]))

def _gpf_core(md,z,gr,valid,grid,glen,vmin,step,gs,ls,ir,p,N,device,gen,dtype=_GDT):
    B,T = md.shape
    AL=p["ANCC_ALPHA"];RN=p["ANCC_RN"];PN=p["ANCC_PN"];IR=p["ANCC_IR"];IS=p["ANCC_IS"];RP=p["ANCC_RP"];RR=p["ANCC_RR"]
    RES=p["PF_RESAMP"];RC=p["PF_RATE_CLIP"];GP=p["PF_GR_POWER"];JP=p["PF_JUMP_PROB"];JS=p["PF_JUMP_STD"];CL=p["PF_TVT_CLIP_MARGIN"]
    UP=bool(p.get("PF_USE_PHYS",False));PS=float(p.get("PF_PHYS_SIG",1.0))
    def rn(): return torch.randn((B,N),generator=gen,device=device,dtype=dtype)
    pos=ls[:,None]+IS*rn(); rate=ir[:,None]+IR*rn(); w=torch.full((B,N),1.0/N,device=device,dtype=dtype)
    log_lik=torch.zeros(B,device=device,dtype=torch.float64); pts=torch.zeros((B,T),device=device,dtype=dtype)
    tvt_lo=vmin-CL; tvt_hi=vmin+(glen.to(dtype)-1)*step+CL
    glast=torch.gather(grid,1,(glen-1).clamp_min(0).unsqueeze(1)); g0=grid[:,0:1]
    pm=md[:,0]-1.0; arN=torch.arange(N,device=device,dtype=dtype); zero=torch.zeros((),device=device,dtype=dtype); dipp=ir[:,None]
    for i in range(T):
        act=valid[:,i]; cur=md[:,i]; dm=(cur-pm).clamp_min(1.0)
        rate_n=AL*rate+RN*rn()
        if RC>0: rate_n=rate_n.clamp(-RC,RC)
        pos_n=pos+rate_n*dm[:,None]+PN*rn()
        if JP>0 and JS>0:
            jm=torch.rand((B,N),generator=gen,device=device,dtype=dtype)<JP; pos_n=pos_n+torch.where(jm,JS*rn(),zero)
        zi=z[:,i][:,None]; tvt=(pos_n-zi).clamp(tvt_lo[:,None],tvt_hi[:,None]); pos_n=tvt+zi; am=act[:,None]
        pos=torch.where(am,pos_n,pos); rate=torch.where(am,rate_n,rate)
        gri=gr[:,i]; obs=act & ~torch.isnan(gri); v=pos-zi
        ii=(v-vmin[:,None])/step[:,None]; i0=torch.floor(ii).long()
        below=i0<0; above=i0>=(glen-1)[:,None]; i0c=i0.clamp_min(0); i0c=torch.minimum(i0c,(glen-2).clamp_min(0)[:,None])
        t=ii-i0c.to(dtype); gA=torch.gather(grid,1,i0c); gB=torch.gather(grid,1,i0c+1); eg=gA*(1-t)+gB*t
        eg=torch.where(below,g0,eg); eg=torch.where(above,glast,eg)
        d=((gri[:,None]-eg)/gs[:,None]).abs(); dp=d**GP
        lk=torch.where(dp<600.0,torch.exp(-0.5*dp),zero).clamp_min(1e-300)
        avg=(w*lk).sum(1)
        log_lik=log_lik+torch.where(obs,torch.log(avg.clamp_min(1e-300).double()),torch.zeros((),device=device,dtype=torch.float64))
        if UP:
            dph=(rate-dipp)/PS; lkp=torch.exp(-0.5*dph*dph).clamp_min(1e-300)
        else: lkp=torch.ones_like(w)
        wn=torch.where(obs[:,None],w*lk*lkp,w*lkp); ws=wn.sum(1,keepdim=True)
        w=torch.where(ws>0,wn/ws.clamp_min(1e-300),torch.full_like(w,1.0/N))
        ne=(w*w).sum(1); need=act & ((1.0/ne)<(RES*N))
        if bool(need.any()):
            cumw=torch.cumsum(w,1); u0=torch.rand((B,1),generator=gen,device=device,dtype=dtype)*(1.0/N); u=u0+arN[None,:]/N
            idx=torch.searchsorted(cumw,u,right=False).clamp(0,N-1)
            pos_rs=torch.gather(pos,1,idx)+RP*rn(); rate_rs=torch.gather(rate,1,idx)+RR*rn()
            if RC>0: rate_rs=rate_rs.clamp(-RC,RC)
            nm=need[:,None]; pos=torch.where(nm,pos_rs,pos); rate=torch.where(nm,rate_rs,rate); w=torch.where(nm,torch.full_like(w,1.0/N),w)
        pts[:,i]=(w*(pos-zi)).sum(1); pm=torch.where(act,cur,pm)
    return pts, log_lik

def _gpad(inps, device, dtype=_GDT):
    W=len(inps); Tmax=max(len(x["md"]) for x in inps); Gmax=max(len(x["gg"]) for x in inps)
    md=torch.zeros((W,Tmax),dtype=dtype); z=torch.zeros((W,Tmax),dtype=dtype)
    gr=torch.full((W,Tmax),float("nan"),dtype=dtype); valid=torch.zeros((W,Tmax),dtype=torch.bool)
    grid=torch.zeros((W,Gmax),dtype=dtype); glen=torch.zeros(W,dtype=torch.long)
    vmin=torch.zeros(W,dtype=dtype); step=torch.zeros(W,dtype=dtype); gs=torch.zeros(W,dtype=dtype)
    ls=torch.zeros(W,dtype=dtype); ir=torch.zeros(W,dtype=dtype)
    for b,x in enumerate(inps):
        T=len(x["md"]); G=len(x["gg"])
        md[b,:T]=torch.from_numpy(x["md"].astype("float32"));
        if T<Tmax: md[b,T:]=md[b,T-1]
        z[b,:T]=torch.from_numpy(x["z"].astype("float32"))
        gr[b,:T]=torch.from_numpy(x["gr"].astype("float32")); valid[b,:T]=True
        grid[b,:G]=torch.from_numpy(x["gg"].astype("float32"))
        if G<Gmax: grid[b,G:]=grid[b,G-1]
        glen[b]=G; vmin[b]=x["gmin"]; step[b]=x["gst"]; gs[b]=x["gs"]; ls[b]=x["ls"]; ir[b]=x["ir"]
    to=lambda t: t.to(device)
    return dict(md=to(md),z=to(z),gr=to(gr),valid=to(valid),grid=to(grid),glen=to(glen),
                vmin=to(vmin),step=to(step),gs=to(gs),ls=to(ls),ir=to(ir))

def _grun_scales(inps,p,S,device,seed,chunk,scales):
    N=int(p["ANCC_N"]); W=len(inps); out=[None]*W
    def _ws(base,pr,sc):
        ww=_np.exp(_np.where(_np.isfinite(base),base,-_np.inf)/max(float(sc),1e-6))
        ww=ww/ww.sum() if ww.sum()>1e-300 else _np.full(len(base),1.0/len(base)); return (ww[:,None]*pr).sum(0).astype(_np.float32)
    for c in range(0,W,chunk):
        sub=inps[c:c+chunk]; w=len(sub); pad=_gpad(sub,device); rep=lambda t: t.repeat_interleave(S,dim=0)
        gen=torch.Generator(device=device); gen.manual_seed(seed)
        pts,ll=_gpf_core(rep(pad["md"]),rep(pad["z"]),rep(pad["gr"]),rep(pad["valid"]),rep(pad["grid"]),
                         pad["glen"].repeat_interleave(S),rep(pad["vmin"]),rep(pad["step"]),
                         rep(pad["gs"]),rep(pad["ls"]),rep(pad["ir"]),p,N,device,gen)
        pts=pts.view(w,S,-1).double().cpu().numpy(); ll=ll.view(w,S).cpu().numpy()
        del pad
        if device == "cuda": torch.cuda.empty_cache()      # チャンク毎に解放(OOM対策)
        for j in range(w):
            T=len(sub[j]["md"]); pr=pts[j,:,:T]
            base=ll[j]-_np.nanmax(ll[j]) if _np.isfinite(ll[j]).any() else _np.zeros_like(ll[j])
            d={f"pf_scale_{sc:g}": _ws(base,pr,sc) for sc in scales}
            d["pf_mean"]=_np.mean([d[f"pf_scale_{sc:g}"] for sc in scales],0).astype(_np.float32)
            out[c+j]=d
    return out
print("v20 GPU PF engine ready | r0 phys=%s seeds=%d" % (_GP["PF_USE_PHYS"], _GN_SEEDS))

In [ ]:
# ===== セル2: test 全坑井を GPU PF で前計算 → run_pf_lik_ensemble_scales を差し替え =====
def _gfp(hw):
    md = hw["MD"].to_numpy(float)
    return (len(hw), round(float(md[0]),1), round(float(md[-1]),1))

# データパス: 7.191 の CFG.dataset_path を優先、無ければ自動探索
try:    _COMP = str(CFG.dataset_path)
except Exception:
    import glob as _g2
    _s = _g2.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    _COMP = _os.path.dirname(_s[0]) if _s else "/kaggle/input/competitions/rogii-wellbore-geology-prediction"

_test_files = sorted(_glob.glob(_os.path.join(_COMP,"test","*__horizontal_well.csv")))
print("test wells:", len(_test_files), "| COMP:", _COMP)
_inps=[]; _fps=[]
for _hp in _test_files:
    _hw=_pd.read_csv(_hp); _tw=_pd.read_csv(_hp.replace("__horizontal_well","__typewell"))
    _x=_gbuild(_hw,_tw,_GP)
    if _x is None: continue
    _inps.append(_x); _fps.append(_gfp(_hw))
_t0=_time.perf_counter()
_GCACHE={}
if _inps:
    _res=_grun_scales(_inps,_GP,_GN_SEEDS,_GDEV,0,_GCHUNK,SELECTOR_SCALES)
    for _fp,_d in zip(_fps,_res): _GCACHE[_fp]=_d
print(f"GPU PF precomputed {len(_GCACHE)} wells in {_time.perf_counter()-_t0:.1f}s ({_GDEV})")

# 差し替え(同名・同戻り値=★坑井全長配列。既知行=TVT_input, 評価行=GPU PF予測)。
# 長さ不一致/未ヒットは安全に原版CPUへフォールバック。
_orig_pf_scales = run_pf_lik_ensemble_scales
def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
    _d = _GCACHE.get(_gfp(hw))
    _ev = hw[hw["TVT_input"].isna()]
    if _d is None or len(_ev) == 0:
        return _orig_pf_scales(hw, tw, scales=scales, n_particles=n_particles, n_seeds=n_seeds)
    _base = hw["TVT_input"].values.astype(float)        # 全長(既知=値, 評価=NaN)
    _idx = list(_ev.index)
    def _full(vec):
        if vec is None or len(vec) != len(_idx): return None
        o = _base.copy(); o[_idx] = vec; return o
    out = {}
    for _k, _v in _d.items():
        if not _k.startswith("pf_scale_"): continue
        _f = _full(_v)
        if _f is None:                                  # 長さズレ→安全にCPU
            return _orig_pf_scales(hw, tw, scales=scales, n_particles=n_particles, n_seeds=n_seeds)
        out[_k] = _f
    _fm = _full(_d["pf_mean"])
    if _fm is None:
        return _orig_pf_scales(hw, tw, scales=scales, n_particles=n_particles, n_seeds=n_seeds)
    out["pf_mean"] = _fm
    return out
print("OK: run_pf_lik_ensemble_scales -> v20 GPU PF(全長配列で返す / selector のみ差し替え / run_pf_ancc は原版)")